# Tutorial 3: Machine learning methods

_Luuk Kempen, Marius Juul Nielsen, Mie Andersen_

In [ ]:
try:
    import ics26
except ImportError:
    !uv pip install "ics26 @ git+https://github.com/lkkmpn/intercat-school-2026.git"

In [ ]:
!curl -fsSL https://github.com/lkkmpn/intercat-school-2026/raw/refs/heads/main/files/ml-data.tar.gz | tar xz

In [ ]:
from ics26.ml_methods import setup_notebook
from ics26.utils import Timer, copy_atoms
from ase.atom import Atom
from ase.atoms import Atoms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np

setup_notebook()

## Atomic structure of a surface

In this tutorial, you will be working with hydrogen on top of a gold surface.
Real surfaces contain an enormous amount of atoms, far too many to simulate
directly. But because the gold is crystalline, and crystal surfaces are
periodic, the same pattern repeats over and over again. We only need to simulate
a small piece that repeats.

Let's start with a stepped gold(211) surface. We first read the file and
visualize the structure. This visualization shows the surface from the top down;
lighter atoms have a higher $z$ coordinate than darker atoms. The red rectangle
shows the unit cell we are simulating, and the yellow atoms in this rectangle
are the atoms we are simulating. The faded atoms around it are the periodic
images; they are the same atoms as the yellow ones, just repeated to show the
surface periodicity.

In [ ]:
from ase.io import read
from ics26.ml_methods.plot import plot_atoms

gold_stepped = read('gold_stepped.traj')
plot_atoms(gold_stepped)

The step in the surface is also visible if we visualize the structure from a
front perspective. This also shows that we only simulate a few layers of the
surface. Again, we cannot simulate the immense number of bulk layers that would
be present in reality. Instead, because atoms on the top of the surface mostly
only feel their neighbors, we only need to simulate a few layers.

In [ ]:
plot_atoms(gold_stepped, plane='xz')

<hr>

## From atomic structure to a descriptor

Because the input to a machine learning potential should respect molecular
symmetries (permutation, translation, rotation), we cannot just feed in the
atomic positions. Hence, we first have to transform the atomic structure to a
descriptor. In this part of the tutorial, you will learn how to create a SOAP
(smooth overlap of atomic positions) descriptor for an atomic structure, and
explore its properties.

For this exploration, we introduce an atomic hydrogen adsorbate on top of the
gold surface.

In [ ]:
gold_stepped_hydrogen = read('gold_stepped_hydrogen.traj')
plot_atoms(gold_stepped_hydrogen)

The SOAP descriptor provides a vector for each atom, which describes that atom's
local environment. A Gaussian is placed on each neighbor atom within a cutoff
radius $r_{\mathrm{cut}}$, leading to an atomic density field. This field is
expanded in radial basis functions and spherical harmonics, whose coefficients
are used to obtain the vector. Later on, you will see that this vector respects
atomic symmetries.

To create a SOAP descriptor, we need to decide the cutoff radius, the number of
radial basis functions $n_{\mathrm{max}}$ to include, and the maximum degree of
spherical harmonics $l_{\mathrm{max}}$ to include. The larger the latter two
parameters, the larger the descriptor vector, making it more expressive but also
more expensive. (There are additional parameters one can choose, but we will
not go into those.)

In this tutorial, we will use a relatively simple SOAP descriptor, with
$r_{\mathrm{cut}} = 5\,\mathrm{Å}$, $n_{\mathrm{max}} = 2$, and
$l_{\mathrm{max}} = 3$.

Because SOAP evaluates the atomic density field per element, we also need to
specify beforehand which elements we have in our system. In this case, that is
gold (Au) and hydrogen (H).

You can get the size of the descriptor (the number of features) by calling
`.get_number_of_features()` on the `SOAP` object.

In [ ]:
from dscribe.descriptors.soap import SOAP

species = ['Au', 'H']
r_cut = 5.0  # Å
n_max = 2
l_max = 3

soap = SOAP(species=species, r_cut=r_cut, n_max=n_max, l_max=l_max, periodic=True)

print(f'Number of features: {soap.get_number_of_features()}')

You can play around with $n_{\mathrm{max}}$ and $l_{\mathrm{max}}$ to see how
the number of features becomes larger as you increase these values.

The formula for the number of features is
$$
N = \frac{n_{\mathrm{species}} \, n_{\mathrm{max}} \, (n_{\mathrm{species}} \, n_{\mathrm{max}} + 1)}{2} (l_{\mathrm{max}} + 1);
$$
check that this matches the number of features you see.

If you did play around with the values, we recommend setting them back to the
defaults (`r_cut = 5.0`, `n_max = 2`, and `l_max = 3`) to ensure the notebook
runs correctly.

<hr>

To create a descriptor, we call `.create()` on the `SOAP` object.

We will only explore the SOAP vector for the hydrogen atom. By default, the SOAP
code will create a vector for all atoms and return a matrix. With the `centers`
argument, you can specify the indices of atoms for which to evaluate the SOAP
vector. Because the hydrogen atom is the last atom in the structure, you can
provide the index `-1`.

When you provide a single `Atoms` object, `soap.create()` returns a matrix of
size `(n_centers, n_features)`. You can extract the vector for hydrogen by
simply selecting the first element of this matrix.

In [ ]:
soap_h = soap.create(gold_stepped_hydrogen, centers=[-1])[0]

print('SOAP vector for hydrogen:')
print(soap_h)
print(f'Shape of SOAP vector: {soap_h.shape}')

Check that the shape of the SOAP vector matches the number of features reported
by SOAP above.

<hr>

Now, you will see how the SOAP vector describes the atomic local environment
while respecting symmetries.

We can move the hydrogen atom over by one gold atom in the $y$ direction, and
the local environment doesn't change, due to symmetry in the surface. Convince
yourself that this is true by visualizing the images.

In [ ]:
# The file we're loading contains 6 images, all with the hydrogen atom moved up
# by one gold atom compared to the previous image.
# Note that after the third image, the simulated hydrogen atom moves to the
# bottom, due to the periodic boundary conditions!

# `index=':'` means that all images from the file are read into a list
gold_stepped_hydrogen_y = read('gold_stepped_hydrogen_y.traj', index=':')
plot_atoms(gold_stepped_hydrogen_y)

Now, for each image, create the SOAP vector and add it to a list.

In [ ]:
soap_vectors = []

for image in gold_stepped_hydrogen_y:
    soap_h = soap.create(image, centers=[-1])[0]  # YOUR CODE HERE
    soap_vectors.append(soap_h)

soap_vectors = np.array(soap_vectors)
print(soap_vectors)

You should be able to manually inspect that each row is the same. You can also
check this by calculating the distance between the first SOAP vector and all
SOAP vectors.

In [ ]:
for i in range(6):
    distance = np.linalg.norm(soap_vectors[i] - soap_vectors[0])
    print(f'Distance between B1 and B{i+1}: {distance:.4g}')

Note that these distances will not be exactly zero; computers are not infinitely
precise. However, a distance on the order of $10^{-13}$ is basically negligible.

<hr>

Next, we can move the hydrogen atom in the $x$ direction. Due to the step in the
surface, in this case the local environment _does_ change.

In [ ]:
# this file contains 6 images, all with the hydrogen moved to the right by one gold atom

gold_stepped_hydrogen_x = read('gold_stepped_hydrogen_x.traj', index=':')
plot_atoms(gold_stepped_hydrogen_x)

Can you recognize the periodicity? Which image is identical to the first?

<hr>

Again, create the SOAP vector for each image.

In [ ]:
soap_vectors = []

for image in gold_stepped_hydrogen_x:
    soap_h = soap.create(image, centers=[-1])[0]  # YOUR CODE HERE
    soap_vectors.append(soap_h)

soap_vectors = np.array(soap_vectors)
print(soap_vectors)

If you look closely, you should now be able to see that the SOAP vectors are no
longer the same. This is more obvious by again calculating the distance between
SOAP vectors.

In [ ]:
for i in range(6):
    distance = np.linalg.norm(soap_vectors[i] - soap_vectors[0])
    print(f'Distance between A1 and A{i+1}: {distance:.4g}')

Did you correctly recognize the periodicity?

<hr>

These distances show that, in SOAP space, the second image is more different to
the first image than the third one is to the first. Furthermore, the fourth
image is identical to the first image, as the step repeats every three gold
atoms.

Comparing these SOAP vectors by looking at 40 (or more) numbers is quite
impractical. We can convert these 40-dimensional vectors into two-dimensional
vectors using **dimensionality reduction** (in this case, using **principal
component analysis**, PCA). You don't need to know the details of this (we have
written the functions that deal with this), just that only the most important
contributions to the SOAP vector remain.

The code below shows how you can transform the six SOAP vectors generated
earlier into two-dimensional vectors, useful for plotting.

In [ ]:
from ics26.ml_methods.dimensionality_reduction import setup_reduction

reduction = setup_reduction(soap)

print(reduction.transform(soap_vectors))

And with the helper function `plot_reduction()`, you can also plot these
vectors.

In [ ]:
from ics26.ml_methods.plot import plot_reduction

# we plot only the first three, because the next three are the same as the first
labels = ['A1', 'A2', 'A3']
transformed = reduction.transform(soap_vectors[:3])

plot_reduction(
    reduction=reduction,
    transformed_vectors=transformed,
    labels=labels
)

<hr>

Another nice property of SOAP descriptors is that they are **smooth**: if an
atom only moves a little bit, the SOAP descriptor also only changes slightly.
You will explore this property by looking at **diffusion** of the hydrogen atom
on the stepped gold surface.

We have prepared a set of images in which the hydrogen atom diffuses between the
six positions you have already been looking at. You can visualize the images
below; each 20th image is at one of the positions you saw earlier.

In [ ]:
from ics26.ml_methods.plot import plot_atoms_slider

gold_stepped_hydrogen_x_diffusion = read('gold_stepped_hydrogen_x_diffusion.traj', index=':')

plot_atoms_slider(gold_stepped_hydrogen_x_diffusion)

<hr>

Now, calculate the hydrogen SOAP vectors for all of these diffusion images, and
transform them to two-dimensional vectors using the `reduction` object.

In [ ]:
soap_vectors_diffusion = []

for image in gold_stepped_hydrogen_x_diffusion:
    soap_h = soap.create(image, centers=[-1])[0]  # YOUR CODE HERE
    soap_vectors_diffusion.append(soap_h)

soap_vectors_diffusion = np.array(soap_vectors_diffusion)

transformed_diffusion = reduction.transform(soap_vectors_diffusion)  # YOUR CODE HERE

print(soap_vectors_diffusion.shape)  # should print (120, n_features)
print(transformed_diffusion.shape)   # should print (120, 2)

<hr>

We can now visualize the reduced SOAP vectors next to the images. The three
vectors you calculated earlier for A1, A2, and A3 are provided as reference
vectors, so they still show up as colored arrows in the figure. New is the black
arrow, which shows the reduced SOAP vector for the image visible on the left.

Play around with this visualization, and notice how the SOAP vector smoothly
transitions from A1 to A2 to A3 and back, moving away from these states while
transitioning between them.

In [ ]:
from ics26.ml_methods.plot import plot_atoms_and_reduction_slider

plot_atoms_and_reduction_slider(
    atoms_list=gold_stepped_hydrogen_x_diffusion,
    reduction=reduction,
    transformed_vectors=transformed_diffusion,
    reference_vectors=transformed,
    reference_labels=labels
)

<hr>

## From descriptor to a model

You have seen that a SOAP descriptor is a useful input to a machine learning
model: it is smooth and respects atomic symmetries. Now you will actually train
a model that uses these SOAP descriptors as inputs and predicts the energy of a
hydrogen atom on a gold surface.

To make the problem a bit more interesting, we have modified the stepped gold
surface by introducing a small kink in one fo the step edges, by removing two
atoms from the step.

We have prepared a dataset for you in which a single hydrogen atom has been
placed at many different positions on and above the surface. Each structure
comes with its potential energy, which we have precomputed. All energies are
given relative to the clean surface plus an isolated hydrogen atom, so they can
be seen as adsorption energies: negative values mean the hydrogen prefers being
adsorbed on this site.

Before building a model, it is worth looking at what we have. The figure below
allows you to explore the dataset, showing each location where the hydrogen was
placed and the corresponding adsorption energy value.

In [ ]:
training_data = read('training_data.traj', index=':')
plot_atoms_slider(training_data)

The cell below will generate a histogram of the adsorption energies in the
dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

energies = [atoms.get_potential_energy() for atoms in training_data]
bins = np.arange(-3.2, 2.3, 0.1)

ax.hist(energies, bins=bins)

ax.set_xlabel('Adsorption energy (eV)')
ax.set_ylabel('Count');

What is approximately the lowest adsorption energy in the dataset? Are there
regions which are less-well covered by the dataset?

<hr>

In this tutorial, we will use a **Gaussian process regression (GPR)** model. The
idea here is that the energy of a new structure is predicted from its similarity
to the structures in the training set. This similarity is measured through the
SOAP vectors: structures with similar SOAP vectors (as we explored earlier) are
expected to have similar energies.

The model has a few **hyperparameters**: the kernel length scale, setting how
quickly similarity decays, and a noise term, setting how closely the model fits
the training data. We don't worry about these here; we can use sensible initial
values and let the model optimize them.

Let's first prepare the inputs to the model, starting with the SOAP matrices.
In this case, we want the SOAP vector for all of the atoms (both gold surface
and hydrogen), so we do not have to specify `centers`. Furthermore, we can
provide a list of `Atoms` objects. This gives a three-dimensional object of
shape `(n_structures, n_centers, n_features)`.

In [ ]:
with Timer() as time:
    soaps = soap.create(training_data)

print(f'Created SOAP matrices in {time} s')

print(soaps.shape)

Another input for the model is the energies the model is supposed to learn.
Because the energies are attached to the `Atoms` objects, we can get them
through `atoms.get_potential_energy()`. We then convert this to a NumPy array
to make it easier to work with.

In [ ]:
energies = np.array([atoms.get_potential_energy() for atoms in training_data])

print(energies.shape)

<hr>

To get a fair idea of how good a model is after training, we must test it on
data it has not seen. Therefore, we make a random split of the dataset before
training, selecting 80% for training and 20% for testing. The model is only
trained on the training dataset, while the testing dataset is used purely for
evaluating the model.

In [ ]:
from sklearn.model_selection import train_test_split

soaps_train, soaps_test, energies_train, energies_test = train_test_split(soaps, energies, train_size=0.8, test_size=0.2, random_state=42)

print(f'Shape of SOAP matrices for training: {soaps_train.shape}')
print(f'Shape of SOAP matrices for testing:  {soaps_test.shape}')
print(f'Shape of energies for training:      {energies_train.shape}')
print(f'Shape of energies for testing:       {energies_test.shape}')

The `train_test_split` function nicely handles this for us. Do the sizes of the
resulting objects make sense?

💡 **Note:** Because we sample train and test configurations from the same
region, a random split probably gives an optimistic picture of the model
performance. A harder test would include configurations that were not at all
in the training set (i.e., keeping the entire area around the kink out of the
training dataset).

<hr>

Now, we can train the GPR model on the training split.

In [ ]:
from ics26.ml_methods.gpr import SGPR

with Timer() as time:
    gpr = SGPR()
    gpr.fit(descriptors=soaps_train, energies=energies_train, optimize=False)

print(f'Trained GPR model in {time} s')

<hr>

Let's see how good the model is.

For this, we can create a **parity plot**: a scatterplot with one point for each
structure, with the target energy on the $x$ axis and the predicted energy on
the $y$ axis. Perfect predictions follow the diagonal line $y = x$. We can also
report error metrics such as the **mean absolute error (MAE)** or the **root
mean square error (RMSE)**:

$$
\begin{align*}
\mathrm{MAE} &= \frac{1}{n} \sum_{i=1}^n \left| \hat{E}_i - E_i \right|; \\
\mathrm{RMSE} &= \sqrt{\frac{1}{n} \sum_{i=1}^n \left( \hat{E}_i - E_i \right)^2},
\end{align*}
$$
where $\hat{E}_i$ is the predicted energy of structure $i$ and $E_i$ is the
target energy of structure $i$.

A low error on the training data shows the model can reproduce well what it was
shown. More important is the error on the test data; how well can the model
predict unseen data? This tells you something about whether the model has really
learned something general. If the test error is much larger than the training
error, that is a sign the model might be **overfitting**.

To make such a parity plot and calculate error metrics, we first need to use the
model to predict potential energies for the training and testing splits. We can
do this using the `.predict_energy()` method of the model, providing a SOAP
matrix as input.

In [ ]:
energies_train_predicted = np.array([gpr.predict_energy(soap) for soap in soaps_train])
energies_test_predicted = np.array([gpr.predict_energy(soap) for soap in soaps_test])

print(f'Shape of predicted energies for training: {energies_train_predicted.shape}')
print(f'Shape of predicted energies for testing:  {energies_test_predicted.shape}')

Next, we can create the parity plots and calculate the error metrics.

In [ ]:
from ics26.ml_methods.plot import plot_parity

fig, (ax_train, ax_test) = plt.subplots(1, 2, figsize=(8, 4))

plot_parity(
    ax=ax_train,
    targets=energies_train,
    predictions=energies_train_predicted,
    title='Training data'
)
plot_parity(
    ax=ax_test,
    targets=energies_test,
    predictions=energies_test_predicted,
    title='Testing data'
)

In [ ]:
def mae(targets, predictions):
    return np.mean(np.abs(targets - predictions))

def rmse(targets, predictions):
    return np.sqrt(np.mean((targets - predictions) ** 2))

print(f'Training MAE:  {mae(energies_train, energies_train_predicted):.2f} eV')
print(f'Testing  MAE:  {mae(energies_test, energies_test_predicted):.2f} eV')
print(f'Training RMSE: {rmse(energies_train, energies_train_predicted):.2f} eV')
print(f'Testing  RMSE: {rmse(energies_test, energies_test_predicted):.2f} eV')

How do the error metrics look? How do the testing split errors compare to the
training split errors? Which region shows the largest errors?

<hr>

## Using the model

Now, we can use the model to predict energies for structures that we never
calculated. As a simple test, we can place the hydrogen atom above a gold atom,
moving it away in the $z$ direction and evaluating the potential energy as a
function of distance to the atom.

In [ ]:
gold_kinked = read('gold_kinked.traj')

initial_distance = 0.8  # Å
final_distance = 4.0  # Å
n_steps = 17

distances = np.linspace(initial_distance, final_distance, n_steps)
d_energies = []

for distance in distances:
    h_position = gold_kinked.positions[0] + [0, 0, distance]
    image = gold_kinked + Atom('H', position=h_position)
    soap_matrix = soap.create(image)  # YOUR CODE HERE
    energy = gpr.predict_energy(soap_matrix)  # YOUR CODE HERE
    d_energies.append(energy)

fig, ax = plt.subplots(figsize=(5, 4))

ax.plot(distances, d_energies, 'o--')
ax.axhline(0.0, lw=0.8, ls=':', c='k')

ax.set_xlabel('H–Au distance (Å)')
ax.set_ylabel('Adsorption energy (eV)');

What potential energy profile did you expect? How does this one compare?

<hr>

Alternatively, we can place the hydrogen atom at a fixed height above the
surface and scan the predicted **potential energy landscape**.

In [ ]:
from ics26.ml_methods.plot import plot_potential_energy_surface
from tqdm.contrib.itertools import product

n_steps = 20

xs = np.linspace(0, gold_kinked.cell[0, 0], n_steps, endpoint=False)
ys = np.linspace(0, gold_kinked.cell[1, 1], n_steps, endpoint=False)

z = np.max(gold_kinked.positions[:, 2]) + 1.0  # Å

xx, yy = np.meshgrid(xs, ys)
energies_grid = np.zeros(xx.shape)

for i, j in product(range(n_steps), range(n_steps)):
    h_position = [xx[j, i], yy[j, i], z]
    image = gold_kinked + Atom('H', position=h_position)  # YOUR CODE HERE
    soap_matrix = soap.create(image)  # YOUR CODE HERE
    energy = gpr.predict_energy(soap_matrix)  # YOUR CODE HERE
    energies_grid[j, i] = energy

plot_potential_energy_surface(gold_kinked, xx, yy, energies_grid)

Can you explain the potential energy surface?

<hr>

These two applications show the payoff of using a machine learning model: you
were able to perform a few hundred energy calculations in seconds, whereas a
single calculation on a system like this would take minutes/hours with DFT.

However, the way we have been using the model is a bit unwieldy: we had to
manually create the SOAP matrix for each structure and then predict the energy.

An easier method of working with the model is by turning it into an **ASE
calculator**, such that it can be attached to an `Atoms` object and used with
everything else ASE offers, like relaxation or molecular dynamics.

As an example, look back to the example where you calculated the adsorption
energy of a H atom as function of $z$ coordinate. Recreate the structure where
the adsorption energy was lowest (read the distance from the figure you
created), and use the ASE calculator to calculate its energy.

In [ ]:
from ics26.ml_methods.gpr import SOAPGPRCalculator

calc = SOAPGPRCalculator(soap, gpr)

image = gold_kinked + Atom('H', position=gold_kinked.positions[0] + [0, 0, 1.8])  # YOUR CODE HERE
image.calc = calc

print(f'Adsorption energy: {image.get_potential_energy():.2f} eV')

plot_atoms(image)

Does this match the value from the earlier figure?

<hr>

The ASE calculator we created also knows how to calculate the **forces** on the
atoms as the (negative) derivative of energy with respect to position;
$F = -\dfrac{\partial E}{\partial x_i}$.

In [ ]:
print(f'Forces on hydrogen atom: {image.get_forces()[-1]} eV/Å')

As mentioned, this allows us to also set up a **relaxation**, moving the
hydrogen atom to a local minimum by taking steps in the direction of the
gradient. Let's see where this hydrogen atom will move to if we let it relax
to a local minimum. For this, we will use the **BFGS** algorithm, which is one
of many algorithms that can be used to relax a structure.

(Can you already make a prediction based on the forces you just evaluated?)

In [ ]:
from ase.optimize.bfgs import BFGS
from ics26.utils import ImageRecorder

rec = ImageRecorder(image)

dyn = BFGS(image)
dyn.attach(rec)
dyn.run(fmax=0.01);

Let's see where the hydrogen atom ended up.

In [ ]:
plot_atoms_slider(rec.images)

Is this where you predicted?

<hr>

## Using the model for a search

You can see that this relaxation was incredibly fast, again showing the power of
these machine learning potentials. One disadvantage of doing a relaxation is
that it is **local**: it will only give you one local minimum, not the global
minimum. To find this, we would have to try many different positions on the
surface—which is possible, due to the speed of the model.

You will now use the model to answer the question: **where on this surface is
the global minimum for hydrogen?** (In other words, **where does hydrogen prefer
to sit on this surface?**)

A simple way to find out is **random structure search**: place the hydrogen atom
at many random positions above the surface, relax it to its local minimum with
the calculator, and keep the lowest-energy structure.

In steps, we will build up this structure search algorithm.

<hr>

The first step is to write a function that can generate initial random
positions. We want these to match some requirements:

* The $x$ and $y$ coordinates can be anywhere within the cell;
* The $z$ coordinate must be at least above the lowest surface atom, and
  maximally $3\,\mathrm{Å}$ above the highest surface atom (so that there is
  still some attractive force from the surface);
* The hydrogen atom must be at least $1.8\,\mathrm{Å}$ away from the closest
  gold atom (there are some large gaps in the bulk, and we don't want the
  hydrogen atom to be generated within the bulk).

In [ ]:
from ase.geometry import get_distances

def generate_initial_random_positions(surface: Atoms, n_positions: int, random_seed: int = 42):
    zs_surface = np.unique(surface.positions[:, 2])
    z_min = zs_surface[-4]  # the lowest surface atom (in the kink) is the fourth from the top
    z_max = zs_surface[-1]  # the highest surface atom is the first from the top

    x_min = 0.0
    x_max = surface.cell[0, 0]
    y_min = 0.0
    y_max = surface.cell[1, 1]
    
    # initialize a random number generator
    rng = np.random.default_rng(random_seed)

    # generate positions in a while loop, as we might reject positions
    positions = []
    while len(positions) < n_positions:
        # generate random position within our bounds
        position = rng.uniform([x_min, y_min, z_min], [x_max, y_max, z_max])

        # get distances from random position to all other atoms, taking periodic boundary conditions into account
        _, distances = get_distances(surface.positions, [position], surface.cell, surface.pbc)
        if np.min(distances) >= 1.8:
            # only add position if it's far enough from gold atoms
            positions.append(position)

    return positions

We can test our random position code by generating a small set of random
positions and visualizing them.

In [ ]:
positions = generate_initial_random_positions(
    surface=gold_kinked,
    n_positions=10
)

random_initial_images = [gold_kinked + Atom('H', position=position) for position in positions]

plot_atoms_slider(random_initial_images)

<hr>

The next step is to write a function which performs the random structure search:

* Generate random positions using the function just created;
* For each random position:
  * Create an `Atoms` object;
  * Attach the calculator and relax;
  * Compare the relaxed energy to the best (lowest) energy found so far;
  * If the relaxed energy is lower, store that energy (and structure) as the
    best candidate.

In [ ]:
def random_structure_search(surface: Atoms, calc: SOAPGPRCalculator, n_positions: int, random_seed: int = 42) -> tuple[list[Atoms], Atoms]:
    initial_positions = generate_initial_random_positions(surface, n_positions, random_seed)

    best_energy = np.inf
    best_image = None

    all_images: list[Atoms] = []

    for i, position in enumerate(tqdm(initial_positions)):
        image = surface + Atom('H', position=position)  # YOUR CODE HERE
        image.calc = calc
        dyn = BFGS(image, logfile=None)  # `logfile=None` disables printing per-step output
        dyn.run(fmax=0.01)

        image = copy_atoms(image)
        all_images.append(image)

        if image.get_potential_energy() < best_energy:  # YOUR CODE HERE
            best_energy = image.get_potential_energy()
            best_image = image

            tqdm.write(f'Iteration {i}: improved to {best_energy:.2f} eV')

    return all_images, best_image

Now, you can run the structure search.

In [ ]:
rss_images, best_image = random_structure_search(gold_kinked, calc, n_positions=50)

Visualize all the structures you generate, and navigate to the lowest-energy
structure. What does it look like?

In [ ]:
plot_atoms_slider(rss_images)

Look back at the histogram of energies in the training data, and compare this to
the lowest energy you found in this random structure search. Do you think you
can trust this energy?

<hr>

If you didn't change any of the settings, the lowest-energy structure you found
should be one where the hydrogen atom is between four gold atoms at the step,
with an adsorption energy of $-4.79\,\mathrm{eV}$.

However, you should remember that this is just the energy predicted by the
model; it is important to **validate** this energy by re-evaluating this
structure with the original calculator (like DFT). If you do that, you will
find that the true adsorption energy for this structure is only
$-2.36\,\mathrm{eV}$. Because there are structures with lower adsorption
energies in the training dataset, you can be sure that this is not actually the
true global minimum. Clearly, you have found a region of space which is not
well-described by the model.

To explore this in more detail, we have taken your 50 randomly-generated
structures (assuming you didn't change any of the settings), and evaluated
their energies with the true potential.

In [ ]:
rss_images_reevaluated = read('rss_data.traj', index=':')

energies_predicted = [image.get_potential_energy() for image in rss_images]
energies_true = [image.get_potential_energy() for image in rss_images_reevaluated]

fig, ax = plt.subplots(figsize=(4, 4))

plot_parity(
    ax=ax,
    targets=energies_true,
    predictions=energies_predicted,
    title='Random structure search data'
)

You can see the results are not that good, as the model **overbinds** most of
the data points—it predicts a much lower (stronger) adsorption energy than the
true value.

<hr>

As we have now re-evaluated the structures in the true potential, you can use
this data to teach the model about these structures, by adding them to the
training data and retraining the model.

In [ ]:
new_soaps = soap.create(rss_images_reevaluated)
new_energies = np.array(energies_true)

soaps = np.concatenate((soaps, new_soaps))
energies = np.concatenate((energies, new_energies))

soaps_train, soaps_test, energies_train, energies_test = train_test_split(soaps, energies, train_size=0.8, test_size=0.2, random_state=42)

with Timer() as time:
    gpr.fit(descriptors=soaps_train, energies=energies_train, optimize=False)

print(f'Trained GPR model in {time} s')

The retrained model can then be used to again perform the random structure search.

In [ ]:
rss_images, best_image = random_structure_search(gold_kinked, calc, n_positions=50)

In [ ]:
plot_atoms(best_image)

These energies look much more realistic, and if we re-evaluate the best
structure (predicted adsorption energy of $-3.13\,\mathrm{eV}$), we find its
true adsorption energy is $-3.10\,\mathrm{eV}$. Pretty good!

<hr>

The last bit you did, generating new data using a cheap model and then
retraining that model using that new data, is actually one of the core steps in
global optimization: **active learning**. In this case, we did it manually, but
most global optimization algorithms will automate this, in a loop which looks
approximately like this:

1. Generate an initial dataset;
2. Train a model on this initial dataset;
3. Search for new candidates using the model;
4. Select a number of candidates to evaluate with the reference method (here, we
   added all candidates);
5. Add the evaluated candidates to the training dataset, and retrain;
6. Go back to step 3 until the best structure stops changing.

Here, we ran this loop once; in practice, it is repeated hundreds or even
thousands of times.